In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/milk/plsda/milk.csv', sep=';') # local copy of Toledo 2022 dataset
data = data_complete.loc[:, '2.66':'22.62']

# Creating a new column 'Class' based on the condition of the samples in the 'Type' column being 'Authentic'
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '2.66':'22.62'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '2.66':'22.62'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-08 16:59:56,905 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-08 16:59:56,929 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [2]:
from modeling import pls_optimized
# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=4,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]

,LVs,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy CV,Sensitivity CV,Specificity CV,CM CV,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred,X Cum Exp Var,Y Cum Exp Var,X Ind Exp Var,Y Ind Exp Var
0,1,0.634328,0.755952,0.43,"[[43, 57], [41, 127]]",0.492537,0.660714,0.21,"[[21, 79], [57, 111]]",0.626087,0.694444,0.511628,"[[22, 21], [22, 50]]",11.749629,7.425722,11.749629,7.425722
1,2,0.731343,0.827381,0.57,"[[57, 43], [29, 139]]",0.567164,0.714286,0.32,"[[32, 68], [48, 120]]",0.695652,0.791667,0.534884,"[[23, 20], [15, 57]]",29.460291,9.243836,17.710662,1.818114
2,3,0.779851,0.851190,0.66,"[[66, 34], [25, 143]]",0.641791,0.744048,0.47,"[[47, 53], [43, 125]]",0.782609,0.819444,0.720930,"[[31, 12], [13, 59]]",51.124817,11.730577,21.664526,2.486741
3,4,0.820896,0.875000,0.73,"[[73, 27], [21, 147]]",0.619403,0.702381,0.48,"[[48, 52], [50, 118]]",0.765217,0.833333,0.651163,"[[28, 15], [12, 60]]",61.679054,16.883401,10.554237,5.152824


## Definição das Zonas Espectrais

In [3]:
spectral_cuts = [
('Ag La', 2.66, 3.10),
('Ag Lb', 3.10, 3.46),
('Ca', 3.46, 3.92),
('background', 3.92, 6.12),
('Fe', 6.12, 6.68),
('Cu', 6.70, 8.37),
('Zn', 8.37, 9.10),
('Bremsstrahlung', 9.10, 20.06),
('Ag compton', 20.06, 21.62),
('Ag ka', 21.48, 22.62)
]

## VIP, Regression Coefficients e SHAP (como no original)

In [4]:

pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

In [6]:
shap_global_importance = pd.read_csv('shap_milk.csv', sep=';') # loading previously saved shap_unique_df

# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,2.99,0.036791,Ag La
1,22.23,0.027351,Ag ka
2,3.22,0.013527,Ag Lb
3,6.35,0.001931,Fe
4,8.73,0.001928,Zn
5,21.41,0.001883,Ag compton
6,18.45,0.001269,Bremsstrahlung
7,3.74,0.000263,Ca
8,8.24,0.000252,Cu
9,4.84,0.000161,background


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [7]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zona 'Ag La': VE = 84.62%, dim = 18 variáveis
Zona 'Ag Lb': VE = 86.51%, dim = 15 variáveis
Zona 'Ca': VE = 21.64%, dim = 19 variáveis
Zona 'background': VE = 3.44%, dim = 87 variáveis
Zona 'Fe': VE = 12.88%, dim = 23 variáveis
Zona 'Cu': VE = 9.26%, dim = 66 variáveis
Zona 'Zn': VE = 17.17%, dim = 29 variáveis
Zona 'Bremsstrahlung': VE = 30.64%, dim = 429 variáveis
Zona 'Ag compton': VE = 52.31%, dim = 62 variáveis
Zona 'Ag ka': VE = 76.93%, dim = 45 variáveis

Scores DataFrame shape: (268, 10)


In [8]:
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred_cont
)

In [9]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Ag La <= 7.76,Ag La <= 7.76,Ag La <= 7.76,Ag La <= 2.70
1,Ag Lb <= 6.48,Ag Lb <= 6.48,Ag La <= 9.01,Ag La <= 7.76
2,Ag La <= 9.01,Ag Lb <= 2.36,Ag La <= 2.70,Ag Lb <= 6.48
3,Ag Lb <= 2.36,Ag La <= 2.70,Ag Lb <= 6.48,Ag Lb <= 2.36
4,Ag La <= 2.70,Ag La <= 9.01,Ag Lb <= 8.04,Ag La <= 9.01
...,...,...,...,...
57,Cu <= 0.32,Ag La > 7.76,Cu <= -0.64,background <= -0.46
58,Fe > -0.36,background > -1.23,Ag La > 7.76,Zn > -0.39
59,Ca <= 1.81,background > -0.46,Fe <= -0.36,Fe <= -0.36
60,Class_A,Class_A,Class_A,Class_A


In [10]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ag La <= 7.76,15.238107,Ag La,7.76,<=
1,Ag Lb <= 6.48,12.796752,Ag Lb,6.48,<=
2,Ag ka <= -7.06,1.051320,Ag ka,-7.06,<=
3,Bremsstrahlung <= -1.54,0.704951,Bremsstrahlung,-1.54,<=
4,Ag compton <= 6.52,0.690518,Ag compton,6.52,<=
5,Fe > -0.36,0.372494,Fe,-0.36,>
6,Ca > 0.46,0.346512,Ca,0.46,>
7,Cu > 0.32,0.320718,Cu,0.32,>
8,Zn > 0.56,0.262477,Zn,0.56,>
9,background <= -0.46,0.228033,background,-0.46,<=


In [11]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zona 'Ag La': VE = 91.39%, dim = 18 variáveis
Zona 'Ag Lb': VE = 90.27%, dim = 15 variáveis
Zona 'Ca': VE = 27.70%, dim = 19 variáveis
Zona 'background': VE = 3.72%, dim = 87 variáveis
Zona 'Fe': VE = 14.68%, dim = 23 variáveis
Zona 'Cu': VE = 15.48%, dim = 66 variáveis
Zona 'Zn': VE = 21.60%, dim = 29 variáveis
Zona 'Bremsstrahlung': VE = 38.87%, dim = 429 variáveis
Zona 'Ag compton': VE = 52.69%, dim = 62 variáveis
Zona 'Ag ka': VE = 80.83%, dim = 45 variáveis


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ag La <= 7.76,15.238107,Ag La,7.76,<=,105.814886,78.0,0.002963,Ag La <= 105.814886
1,Ag Lb <= 6.48,12.796752,Ag Lb,6.48,<=,75.496450,98.0,0.013050,Ag Lb <= 75.496450
2,Ag La <= 9.01,10.618027,Ag La,9.01,<=,126.903607,239.0,0.004801,Ag La <= 126.903607
3,Ag La <= 2.70,9.734752,Ag La,2.70,<=,34.935039,86.0,0.021418,Ag La <= 34.935039
4,Ag Lb <= 2.36,8.027932,Ag Lb,2.36,<=,30.578162,0.0,0.008013,Ag Lb <= 30.578162
...,...,...,...,...,...,...,...,...,...
57,background > 0.40,0.170777,background,0.40,>,4.852293,186.0,0.003287,background > 4.852293
58,Ag La > 7.76,0.167815,Ag La,7.76,>,105.814886,78.0,0.002963,Ag La > 105.814886
59,Fe <= -0.36,0.154303,Fe,-0.36,<=,0.719116,9.0,0.001198,Fe <= 0.719116
60,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [22]:
import plotly.graph_objects as go

n = 1
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ag Lb':
  - Dimensão: 15 variáveis espectrais
  - Range de energias: 3.10 - 3.46
  - Variância explicada pela PC1: 90.27%


In [13]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='regression',
        metric='mean_abs_diff', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
PERTURBATION IMPORTANCE PARA PREDICADOS
Tipo de tarefa (aim): regression
Modo de perturbação: median
Fonte das estatísticas: full
Métrica: mean_abs_diff
Total de folds: 10




,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ag La <= 2.70,Ag La <= 2.70,Ag La <= 2.70,Ag La <= 2.70
1,Ag La <= 7.76,Ag La <= 7.76,Ag La <= 7.76,Ag La <= 7.76
2,Ag Lb <= 2.36,Ag Lb <= 2.36,Ag Lb <= 2.36,Ag Lb <= 2.36
3,Ag La <= 9.01,Ag La <= 9.01,Ag La <= 9.01,Ag La <= 9.01
4,Ag ka > -11.59,Ag ka > -11.59,Ag ka <= -7.06,Ag ka <= -7.06
...,...,...,...,...
57,Ca <= 0.46,background > 0.40,background <= 1.29,background > -1.23
58,background > -1.23,Ca <= 0.46,Ca <= 0.46,Ca <= 0.46
59,Ca <= 1.81,Ca <= 1.81,Ca <= 1.81,Ca <= 1.81
60,Class_A,Class_A,Class_A,Class_A


In [14]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ag La <= 2.70,0.270392
1,Ag La <= 7.76,0.181371
2,Ag Lb <= 2.36,0.163421
3,Ag La <= 9.01,0.152026
4,Ag ka <= -7.06,0.149678
5,Bremsstrahlung <= -1.54,0.128461
6,Ag ka <= 20.95,0.126122
7,Ag ka <= 5.37,0.125895
8,Ag ka > 5.37,0.122808
9,Bremsstrahlung <= 4.15,0.122365


In [15]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ag La <= 2.70,5.762044,Ag La,2.70,<=
1,Ag Lb <= 2.36,3.706879,Ag Lb,2.36,<=
2,Ag ka <= -7.06,2.027356,Ag ka,-7.06,<=
3,Bremsstrahlung <= 9.64,1.556894,Bremsstrahlung,9.64,<=
4,Ag compton <= -1.09,0.865729,Ag compton,-1.09,<=
5,Zn > 0.56,0.095506,Zn,0.56,>
6,Fe > 0.31,0.056914,Fe,0.31,>
7,Ca > 0.46,0.015623,Ca,0.46,>
8,Cu <= 0.32,0.011015,Cu,0.32,<=
9,background > 0.40,0.007564,background,0.40,>


In [16]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ag La <= 2.70,5.762044,Ag La,2.70,<=,34.935039,86.0,0.021418,Ag La <= 34.935039
1,Ag La <= 7.76,4.528915,Ag La,7.76,<=,105.814886,78.0,0.002963,Ag La <= 105.814886
2,Ag Lb <= 2.36,3.706879,Ag Lb,2.36,<=,30.578162,0.0,0.008013,Ag Lb <= 30.578162
3,Ag La <= 9.01,2.963346,Ag La,9.01,<=,126.903607,239.0,0.004801,Ag La <= 126.903607
4,Ag ka <= -7.06,2.027356,Ag ka,-7.06,<=,-250.573937,70.0,0.012824,Ag ka <= -250.573937
...,...,...,...,...,...,...,...,...,...
57,background > -1.23,0.006802,background,-1.23,>,1.242244,67.0,0.007894,background > 1.242244
58,Ca <= 0.46,0.005171,Ca,0.46,<=,3.531024,28.0,0.010826,Ca <= 3.531024
59,Ca <= 1.81,0.002553,Ca,1.81,<=,-2.575534,222.0,0.044016,Ca <= -2.575534
60,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


In [23]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ag La':
  - Dimensão: 18 variáveis espectrais
  - Variância explicada pela PC1: 91.39%


In [18]:
# Permutation importance baseado em mudança nas predições do modelo PLS
# Medimos a média da diferença absoluta entre as predições originais e as predições
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Predições base do modelo PLS
baseline_pred = pls_model.predict(Xcalclass_prep)
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_pred = pls_model.predict(X_perm)
        diffs.append(np.mean(np.abs(baseline_pred - perm_pred)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance': importance_list
})
permutation_df.sort_values(by='Permutation_importance', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance', ascending=False)
permutation_unique_df

,energy,Permutation_importance,Zone
0,22.23,0.058491,Ag ka
1,2.99,0.055478,Ag La
2,3.22,0.026814,Ag Lb
3,8.75,0.013365,Zn
4,6.35,0.011798,Fe
5,21.41,0.009460,Ag compton
6,18.27,0.008225,Bremsstrahlung
7,3.74,0.006772,Ca
8,7.80,0.006045,Cu
9,4.84,0.005265,background


In [19]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'VIP_Score': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_Coefficient' : pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,VIP_Score,Reg_Coefficient,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Ag La,Ag ka,Ag La,Ag ka,Ag La,Ag La
1,Ag Lb,Zn,Ag ka,Ag La,Ag Lb,Ag Lb
2,Ag ka,Ag La,Ag Lb,Ag Lb,Ag ka,Ag ka
3,Zn,Fe,Fe,Zn,Bremsstrahlung,Bremsstrahlung
4,Ag compton,Ag Lb,Zn,Fe,Ag compton,Ag compton
5,Fe,Bremsstrahlung,Ag compton,Ag compton,Zn,Fe
6,Bremsstrahlung,Ag compton,Bremsstrahlung,Bremsstrahlung,Fe,Ca
7,Ca,Ca,Ca,Ca,Ca,Cu
8,background,background,Cu,Cu,Cu,Zn
9,Cu,Cu,background,background,background,background


In [20]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28368\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
14,LRC_perturbation,LRC_covariance,0.955219
3,VIP_Score,LRC_perturbation,0.921296
4,VIP_Score,LRC_covariance,0.913166
1,VIP_Score,Shap,0.824700
10,Shap,LRC_perturbation,0.803812
11,Shap,LRC_covariance,0.795682
6,Reg_Coefficient,Permutation,0.781702
9,Shap,Permutation,0.646027
2,VIP_Score,Permutation,0.550425
12,Permutation,LRC_perturbation,0.503812


In [21]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')